# Demo: Wheel Measurements and Empty-Car Hunting

We adapted our teammate's modeling notebook into a short demo. We make example data, check it, create two plots, and run one logistic regression model.

**All data is made up.** These results only show that the code runs. They are not findings about real equipment. The example keeps the original notebook's 80 mph cutoff to show the classification step.

Follow `INSTALL.md`, then restart the kernel and run all cells. No data download is needed.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Entirely invented values; no partner files or measurements are read.
rng = np.random.default_rng(42)
measurements = ["Flange Width (Sd)", "Flange Height (Sh)", "qR", "Flange Width",
                "Max Hollow Wear", "Flange Radius", "Max Flange Angle",
                "Min Flange Root Radius", "Max Hollow Wear - AAR"]
rows = []
for car in range(1, 21):
    for truck in (1, 2):
        for rail in range(1, 8):
            # Alternate outcomes within every car so both split subsets have both classes.
            hunted = (car + truck + rail) % 2 == 0
            speed = float(rng.uniform(55, 75) if hunted else rng.uniform(82, 95))
            for axle in (2 * truck - 1, 2 * truck):
                row = {"Car": car, "Truck": truck, "Axle": axle, "Rail_Profile": rail,
                       "Empty_Truck_Hunted": hunted,
                       "Empty Truck from Acc HS [mph]": speed,
                       "Min Empty Truck from Acc HS [mph]": speed,
                       "Eq Con 3mm": float(rng.uniform(0.1, 0.5))}
                for measurement in measurements:
                    for side in ("L", "R"):
                        row[measurement + "_" + side] = float(rng.uniform(0.1, 2.0))
                rows.append(row)
final_data = pd.DataFrame(rows)
assert len(final_data) == 560
assert not final_data.isna().any().any()
assert final_data.groupby(["Car", "Truck", "Rail_Profile"])["Axle"].nunique().eq(2).all()
print("Synthetic demo data: 560 rows, 20 invented cars, 280 truck/rail assessments.")
print("No confidential data files are required or read.")
final_data.head()


## Make two example plots

We compare hunting rates by rail profile and hollow wear by hunting outcome. The values are random, so these plots are examples.


In [ ]:
# Use the same feature calculations as our teammate's notebook.
final_data["Hollow_Wear_Avg"] = (final_data["Max Hollow Wear - AAR_L"] + final_data["Max Hollow Wear - AAR_R"]) / 2
final_data["qR_Avg"] = (final_data["qR_L"] + final_data["qR_R"]) / 2
final_data["Flange_Width_Avg"] = (final_data["Flange Width_L"] + final_data["Flange Width_R"]) / 2
final_data["Flange Width (Sd)_Avg"] = (final_data["Flange Width (Sd)_L"] + final_data["Flange Width (Sd)_R"]) / 2
final_data["Flange_Angle_Avg"] = (final_data["Max Flange Angle_L"] + final_data["Max Flange Angle_R"]) / 2
final_data["Truck_Hunted"] = final_data["Min Empty Truck from Acc HS [mph]"] < 80


In [ ]:
# Compare how often hunting occurs across the seven rail profiles in percentage
rail_hunting = final_data.groupby("Rail_Profile")["Empty_Truck_Hunted"].mean() * 100

print(rail_hunting)
# Visualize differences in hunting frequency across rail profiles
rail_hunting.plot(kind="bar")

plt.xlabel("Rail Profile")
plt.ylabel("Hunting Rate (%)")
plt.title("Empty-Car Hunting Rate by Rail Profile")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Compare wheelset hollow wear between hunting and non-hunting cases
final_data.boxplot(
    column="Hollow_Wear_Avg",
    by="Empty_Truck_Hunted"
)

plt.xlabel("Truck Hunted")
plt.ylabel("Average Hollow Wear")
plt.title("Wheelset Hollow Wear vs Empty-Car Hunting")
plt.suptitle("")
plt.show()

## Try a model

We keep each car in either training or testing so its rows do not appear in both. Then we fit logistic regression and check its predictions.


In [ ]:
# Select physical wheel characteristics and rail profile
features = [
    "Flange_Width_Avg",
    "Flange Width (Sd)_Avg",
    "qR_Avg",
    "Hollow_Wear_Avg",
    "Flange_Angle_Avg",
    "Rail_Profile"
]

X = final_data[features].copy()
y = final_data["Truck_Hunted"].astype(int)

print("Features:", X.columns.tolist())
print("Missing values:", X.isna().sum().sum())
print("Target classes:", sorted(y.unique()))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Keep all data from the same car together to avoid data leakage
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=final_data["Car"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training classes:", y_train.value_counts().to_dict())
print("Testing classes:", y_test.value_counts().to_dict())
assert set(final_data.iloc[train_idx]["Car"]).isdisjoint(set(final_data.iloc[test_idx]["Car"]))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = [
    "Flange_Width_Avg",
    "Flange Width (Sd)_Avg",
    "qR_Avg",
    "Hollow_Wear_Avg",
    "Flange_Angle_Avg"
]

categorical_features = ["Rail_Profile"]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)



In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Predict hunting on unseen cars
y_pred = model.predict(X_test)

# Evaluate predictions
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print()
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Hunting", "Hunting"],
    zero_division=0
))

# Visualize correct and incorrect predictions
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["No Hunting", "Hunting"],
    cmap="Blues"
)

plt.title("Logistic Regression - Test Results")
plt.show()

## What is next

This checks the setup, plots, and modeling steps. It starts with generated data in the merged-table format. We still need to test the separate Excel merge steps and check our findings using approved partner data.


In [ ]:
print("Demo completed successfully: synthetic data, two plots, grouped split, and one model ran.")
